# 03 · 고급 분석: 패널 고정효과  (모듈 4-A)
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/amnotyoung/oda-ai-stats/blob/main/notebooks/03_panel_fe.ipynb)

> 고급 계량 분석도 AI 도움으로 **양쪽 도구에서** 할 수 있다. 국가×연도 패널의 고정효과를
> Python으로, 그리고 STATA로 — **둘 다 같은 결과**. 도구는 우열이 아니라 **환경**으로 고른다.

In [ ]:
import pandas as pd, numpy as np
import statsmodels.formula.api as smf
df = pd.read_csv("https://raw.githubusercontent.com/amnotyoung/oda-ai-stats/main/data/wdi_panel.csv").dropna(subset=["gdp_pc","life_exp"])
df["log_gdp"] = np.log(df["gdp_pc"])
print("국가:", df["economy"].nunique(), "· 연도:", df["year"].nunique(), "· 행:", len(df))

## 문제 — 국가·시대의 교란을 통제하려면? (식별의 문제)
소득과 기대수명은 둘 다 **시간이 가며 함께 오른다.** 단순 상관엔 *국가 고유차*와 *전세계 시대추세*가
뒤섞여 있다. 진짜 효과를 보려면 이 둘을 **고정효과**로 걷어내야 한다 — 임팩트평가의 핵심 아이디어.

### (1) 통제 안 함 — 단순 회귀(pooled)

In [ ]:
pooled = smf.ols("life_exp ~ log_gdp", data=df).fit()
print("pooled         log_gdp =", round(pooled.params["log_gdp"], 2))

### (2) 국가 고정효과 — 국가 고유차 통제

In [ ]:
fe = smf.ols("life_exp ~ log_gdp + C(economy)", data=df).fit()
print("국가 FE        log_gdp =", round(fe.params["log_gdp"], 2), "(국가 고유효과 흡수)")

### (3) **이원 고정효과** — 국가 + 연도 동시 통제 (고급)
전세계가 함께 좋아진 **시대추세**(`C(year)`)까지 빼면 효과가 또 달라진다.

In [ ]:
twfe = smf.ols("life_exp ~ log_gdp + C(economy) + C(year)", data=df).fit()
print("이원 FE(+연도) log_gdp =", round(twfe.params["log_gdp"], 2), "(시대추세까지 흡수)")
print("\n→ 4.59 → 3.54 → 1.26 : 무엇을 통제하느냐에 따라 답이 바뀐다(식별의 문제).")

### (4) STATA에선 — 같은 분석
```stata
encode economy, gen(country_id)
xtset country_id year
xtreg life_exp log_gdp i.year, fe vce(cluster country_id)
```
- `i.year`로 연도효과까지 한 번에. base STATA — **폐쇄망에서 인터넷 없이 실행** ✅  코드 → `stata/05_panel_fe.do`
- 두 도구 계수가 **정확히 일치**(1.262)한다 — 교차검증.

---
✅ **포인트**: 이원 고정효과는 **차분-차분(DiD) 임팩트평가의 엔진**이다(KOICA M&E와 직결).
"무엇을 통제했나"가 결과를 좌우하므로 *설계·검증은 사람 몫* — AI는 코드를 짜고, 판단은 여러분이 한다.